# Lecture 2 · Notebook 1 — Inside a CNN: receptive fields, one encoder, three tasks

**ML Summer School · Large models: CNNs, GNNs, and deep learning applications**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IPMUCD3/a3net_2026/blob/main/Lecture_Day2_Terao/01_cnn_encoder_and_heads.ipynb)
---

## Set up

In [ ]:
# Setup. Nothing here is part of the lecture -- it just makes `mlschool`
# importable (cloning the course repo if we are on Colab) and imports the usual
# suspects. Run it and move on.
REPO = "https://github.com/drinkingkazu/a3net-lecture2.git"
import os, subprocess, sys
try:
    import mlschool
except ModuleNotFoundError:
    here = [os.path.abspath(d) for d in (".", "..", "../..")]
    root = next((d for d in here
                 if os.path.isfile(os.path.join(d, "mlschool", "__init__.py"))), None)
    if root is None:                                   # not inside a checkout: fetch it
        subprocess.run(["git", "clone", "--depth", "1", REPO, "a3net-lecture2"], check=True)
        root = os.path.abspath("a3net-lecture2")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", root])
    sys.path.insert(0, root)

import mlschool as ms
import time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
DEVICE = ms.device()
ms.hello()

In [ ]:
import mlschool as ms
deck = ms.slides.SlideDeck("slides/NB1/cnn")
deck.show(start=0)

## Where we are

Notebook 0 argued that an architecture is a **prior on the structure of your
data**, and showed the payoff. That was the *why*. This notebook is the *how*. Two questions:

**1. Receptive field**: A convolution is local (e.g., 3x3) by construction. Yet a deep CNN can find spatially larger features. We analyze how the receptive field of a CNN is formed.

**2. Multi-task cascade**: In physics we rarely want a single label per event. 
For insstance, we want to know *what* happened (classification), *where* it happened (segmentation), and *how much energy* was deposited (regression) — from the same detector image. The standard answer is a shared **encoder** plus small task-specific **heads**.

Along the way, the two places where physics data breaks the textbook recipe:

- **Class imbalance.** 98 % of our pixels are background. We will test the
  standard remedy, discover it makes things *worse* here, and find out why —
  which turns out to be more useful than the remedy.
- **The loss is part of the optimisation problem, not a scoreboard.** The
  gradient of a squared error grows without bound in the residual, so one
  badly-simulated event can dominate the gradient of an entire batch.

**Runtime:** roughly 8–12 minutes on a Colab T4.

In [ ]:
train = ms.generate_dataset(4000, seed=0, progress=True)
val   = ms.generate_dataset(1000, seed=1)
ms.summarize(train, "train")

## 1. The receptive field, and why it decides your depth

The **receptive field** of a unit is the region of the *input* that can possibly
influence it. For a stack of convolutions and poolings it follows a two-line
recursion. Walking forward through the network, track

- $j$ — the **jump**: how many input pixels apart two neighbouring units are
  (the product of all strides so far),
- $r$ — the **receptive field size**.

For a layer with kernel $k$ and stride $s$:

$$r \leftarrow r + (k-1)\,j, \qquad j \leftarrow j \cdot s$$

Start with $r = 1$, $j = 1$. That is the whole story.

In [ ]:
def receptive_field(layers):
    """layers: list of (name, kernel, stride). Returns a printable table."""
    r, j = 1, 1
    rows = [("input", 1, 1, r, j)]
    for name, k, s in layers:
        r = r + (k - 1) * j
        j = j * s
        rows.append((name, k, s, r, j))
    print(f"{'layer':<12}{'k':>3}{'s':>3}{'receptive field':>18}{'jump':>7}")
    print("-" * 43)
    for name, k, s, r_, j_ in rows:
        print(f"{name:<12}{k:>3}{s:>3}{r_:>15} px{j_:>7}")
    return r


arch = []
for b in range(1, 5):
    arch += [(f"conv{b}a", 3, 1), (f"conv{b}b", 3, 1), (f"pool{b}", 2, 2)]
final_rf = receptive_field(arch)
print(f"\nfinal receptive field: {final_rf} px, on a {ms.SIZE} px image")

Read the table carefully, because it contains the two facts that govern CNN
design.

**Depth is not a free hyperparameter.** Our track-vs-kink task in Notebook 0
needed the network to see a bend, which spans ~15 px; two blocks give a 16 px
receptive field, which is just enough — and that is why a 16 k-parameter network
solved it completely. A task that needs to relate a shower to a track 60 px away may make a network struggle. 

> **Do this for your own problem.** Write down the physical scale of the feature
> that distinguishes your classes, in pixels. Then make sure your receptive field
> comfortably exceeds it. This costs two minutes and saves weeks.

### Measuring it, rather than trusting the arithmetic

The formula is easy to get wrong once you have padding, dilation, or a
non-obvious stride. So measure it: put a gradient on one output unit and see
which input pixels respond.

One subtlety — we probe a **linear stand-in** for the network (same kernels and
strides, average pooling, no ReLU). The receptive field is a property of the
*wiring*, and with ReLUs a randomly-initialised network would show us only the
pixels that happen to be active for one particular input.

In [ ]:
def probe_network(n_blocks):
    layers = []
    cin = 1
    for _ in range(n_blocks):
        layers += [nn.Conv2d(cin, 4, 3, padding=1), nn.Conv2d(4, 4, 3, padding=1),
                   nn.AvgPool2d(2)]
        cin = 4
    return nn.Sequential(*layers)


def measure_rf(net, size=ms.SIZE):
    # Trick: ask autograd which inputs one output unit depends on. Any input pixel
    # with a non-zero gradient is inside that unit's receptive field.
    x = torch.zeros(1, 1, size, size, requires_grad=True)
    y = net(x)
    cy, cx = y.shape[2] // 2, y.shape[3] // 2     # pick the central output unit
    y[0, :, cy, cx].sum().backward()              # gradient of that unit alone
    g = x.grad[0, 0].abs().numpy()                # (size, size) sensitivity map
    ys, xs = np.nonzero(g > 0)
    return g, (xs.max() - xs.min() + 1)           # width of the responding region


fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for n, ax in zip(range(1, 5), axes):
    g, width = measure_rf(probe_network(n))
    ax.imshow(g > 0, cmap="magma", origin="lower")
    ax.set_title(f"{n} block{'s' if n > 1 else ''}\nmeasured RF = {width} px", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("input pixels that can influence one central output unit", y=1.04)
fig.tight_layout(); plt.show()

The measured widths match the table. Note how quickly the field grows: one block
sees a small patch, four blocks see most of the image.

## 2. The cost of seeing further: resolution

Downsampling buys context and spends resolution. After four poolings our
$96\times96$ image is a $6\times6$ feature map. That is fine if you want one
label per event — but our segmentation task needs an answer **per pixel**, and
a $6\times6$ map cannot tell you which of 256 pixels in its footprint was a
track.

This is a genuine tension, and it has a standard resolution:

- go **down** to build features with large receptive fields (what is here?),
- go back **up** to restore resolution (where exactly is it?),
- and connect the two with **skip connections** that carry the high-resolution
  detail the downsampling path threw away.

That is a **U-Net**, and it remains the default architecture for per-pixel tasks
in physics a decade after it was proposed for biomedical images.

> **Two different things are called "skip connections", and conflating them
> causes confusion.**
>
> - **U-Net skips** are *concatenations* across a resolution gap. Their job is to
>   restore spatial detail. They connect the encoder to the decoder.
> - **Residual skips** (`x + f(x)`, as in ResNet) are *additions* at the same
>   resolution. Their job is to let gradients reach early layers so that deep
>   networks train at all. We build those in Notebook 2.
>
> Same name, different purpose. This notebook uses the first kind; the next uses
> the second.

## 3. One encoder, three heads

Here is the object we will build. A single encoder produces a pyramid of feature
maps; three small heads consume them.

```
                        ┌────────────► classification head  → 3 logits
input 1×96×96           │                (global max-pool)
   │                    │
   ▼                    │
 enc1 16×96×96 ──────┐  ├────────────► energy head → 1 number
   │ pool            │  │                (global SUM-pool)
 enc2 32×48×48 ────┐ │  │
   │ pool          │ │  │
 enc3 64×24×24 ──┐ │ │  │
   │ pool        │ │ │  │
 bottleneck 96×12×12 ───┘
   │ up          │ │ │
   ├─ concat ────┘ │ │
   │ up            │ │
   ├─ concat ──────┘ │
   │ up              │
   ├─ concat ────────┘
   ▼
 segmentation head → 3 logits × 96 × 96
```

**Why share the encoder at all?** Because the features that let you say "this is
a shower" are the same features that let you say "this *pixel* belongs to a
shower" and "this event deposited 3 GeV". Learning them once is cheaper, and —
more importantly — the segmentation labels act as a very rich training signal
that improves the classification head too. Multi-task learning is often a
regulariser you get for free.

**Why does each head pool differently?** This is not a detail. It is the
inductive bias of the head, and it should match the physics of the target:

| head | pooling | reason |
|---|---|---|
| classification | **max** | "is there a shower *anywhere*?" — an existence question, and (Notebook 0) mean-pooling drowns a 2 % occupancy signal |
| energy | **sum** | energy is **extensive**: two identical showers deposit twice the energy, and summing is the operation that counts. See the caveat below — on a fixed-size image this is a weaker statement than it sounds |
| segmentation | none | the answer is per-pixel, so nothing may be pooled away |

> **A caveat on sum vs. mean, because it is easy to overclaim.** Our images are
> a fixed $96\times96$, so the bottleneck is always $6\times6=36$ cells and
> $\text{mean} = \text{sum}/36$ — a constant the next linear layer simply
> absorbs. On fixed-size inputs the two poolings are the *same model* up to a
> rescaling, and exercise 2 confirms that both reproduce the extensive scaling
> correctly. The choice becomes a genuine prior only when the number of pooled
> elements **varies between examples** — which is exactly the point-cloud setting
> of Notebook 3, where dividing by a per-event hit count really does discard the
> multiplicity. Keep `sum` here because it says what you mean; just do not
> believe it is doing work that it is not.

In [ ]:
def conv_bn_relu(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
    )


class Encoder(nn.Module):
    """Downsampling path. Returns the feature pyramid, finest first."""

    def __init__(self, widths=(16, 32, 64, 96)):
        super().__init__()
        self.blocks = nn.ModuleList()
        cin = 1
        for w in widths:
            self.blocks.append(conv_bn_relu(cin, w))
            cin = w
        self.widths = widths

    def forward(self, x):
        # x: (B, 1, 96, 96). We keep every intermediate map, because the decoder
        # needs them: that list IS the skip connections.
        feats = []
        for i, blk in enumerate(self.blocks):
            if i > 0:                     # no pooling before the first block
                x = F.max_pool2d(x, 2)    # halve both spatial dimensions
            x = blk(x)                    # channels grow as resolution shrinks
            feats.append(x)
        # feats = [(B,16,96,96), (B,32,48,48), (B,64,24,24), (B,96,12,12)]
        return feats


class MultiTaskNet(nn.Module):
    def __init__(self, widths=(16, 32, 64, 96), n_classes=3):
        super().__init__()
        self.encoder = Encoder(widths)
        deep = widths[-1]

        self.cls_head = nn.Sequential(nn.Linear(deep, 64), nn.ReLU(),
                                      nn.Linear(64, n_classes))
        self.energy_head = nn.Sequential(nn.Linear(deep, 64), nn.ReLU(),
                                         nn.Linear(64, 1))

        # decoder: upsample and concatenate the matching encoder feature map
        self.up = nn.ModuleList()
        for i in range(len(widths) - 1, 0, -1):
            self.up.append(conv_bn_relu(widths[i] + widths[i - 1], widths[i - 1]))
        self.seg_out = nn.Conv2d(widths[0], n_classes, 1)

    def forward(self, x):
        feats = self.encoder(x)
        deep = feats[-1]                          # (B, 96, 12, 12) the bottleneck

        # Both event-level heads collapse the two spatial axes to nothing, leaving
        # one vector per event: (B, 96, 12, 12) -> (B, 96) -> (B, n_classes) or (B, 1).
        cls = self.cls_head(deep.amax(dim=(2, 3)))               # global MAX
        energy = self.energy_head(deep.sum(dim=(2, 3)) / 100.0)  # global SUM

        # Decoder: walk back up the pyramid. feats[-2-k] is the encoder map at the
        # resolution we are climbing to, so k=0 pairs the 12x12 bottleneck with the
        # 24x24 encoder map, k=1 pairs 24x24 with 48x48, and so on.
        h = deep
        for k, blk in enumerate(self.up):
            skip = feats[-2 - k]                                 # (B, C_skip, S, S)
            h = F.interpolate(h, size=skip.shape[-2:], mode="nearest")   # match S
            h = blk(torch.cat([h, skip], dim=1))    # concat along CHANNELS, not space
        seg = self.seg_out(h)                       # (B, n_classes, 96, 96)

        # squeeze(1) turns the energy head's (B, 1) into (B,) to match the target.
        return cls, seg, energy.squeeze(1)


net = MultiTaskNet()
n_enc = sum(p.numel() for p in net.encoder.parameters())
n_all = sum(p.numel() for p in net.parameters())
print(f"encoder parameters {n_enc:>10,}")
print(f"total parameters   {n_all:>10,}")
print(f"encoder is {100 * n_enc / n_all:.0f}% of the model — the heads are cheap")

## 4. Preparing inputs and targets

Three preparation steps, each of which is a lesson.

**Inputs are normalised.** Charge runs to ~140; we divide by a global constant so
typical values are $O(1)$. Note we use a *fixed* constant computed from the
training set, not a per-event maximum — for the energy task, dividing each event
by its own maximum would destroy the very information we want to regress.

**Targets are standardised.** The energy target is $O(1\text{–}5)$ while the
cross-entropy losses are $O(1)$. If you do not put your regression target on a
comparable scale, the relative weighting of your multi-task loss becomes an
accident of your unit system.

**Everything is a tensor of known dtype.** `seg` is `int8` in the simulator to
save memory; `F.cross_entropy` requires `int64`. Silent dtype bugs are a
depressingly common cause of "the model does not learn".

In [ ]:
CHARGE_SCALE = float(np.percentile(train["image"][train["image"] > 0], 99))
E_MEAN, E_STD = float(train["energy"].mean()), float(train["energy"].std())
print(f"charge scale (99th pct of non-zero) = {CHARGE_SCALE:.1f}")
print(f"energy mean/std = {E_MEAN:.2f} / {E_STD:.2f}")


def prepare(ds):
    return {
        "x": torch.tensor(ds["image"])[:, None] / CHARGE_SCALE,
        "cls": torch.tensor(ds["label"]),
        "seg": torch.tensor(ds["seg"]).long(),
        "energy": torch.tensor((ds["energy"] - E_MEAN) / E_STD),
    }


TR, VA = prepare(train), prepare(val)
print({k: tuple(v.shape) for k, v in TR.items()})

## 5. The imbalance problem, stated numerically

Before training anything, work out what a useless model scores. This is the
physics habit of computing your background rate before claiming a discovery, and
it is just as necessary here.

In [ ]:
counts = np.bincount(train["seg"].ravel(), minlength=3)
frac = counts / counts.sum()
for c in range(3):
    print(f"  {ms.SEG_NAMES[c]:<12}{counts[c]:>12,} pixels   {100 * frac[c]:6.2f} %")
print(f"\n'predict background everywhere' scores {100 * frac[0]:.2f} % pixel accuracy")
print("...while finding exactly zero particles.")

inv = 1.0 / frac
class_weights = torch.tensor(inv / inv.mean(), dtype=torch.float32)
print(f"\ninverse-frequency class weights: "
      f"{', '.join(f'{ms.SEG_NAMES[c]}={class_weights[c]:.1f}' for c in range(3))}")

**Accuracy is the wrong metric**. We will report per-class **recall** (of the true track pixels, how many did we find?), **precision** (of the pixels we called track, how many really were?), and intersection-over-union (**IoU**), the standard segmentation metric precisely because it cannot be inflated by a large true-negative population.


### Class imbalance?
The received wisdom about the *loss* goes like this: unweighted cross-entropy
averages over pixels, so 98 % of the gradient comes from background; the cheapest
way to lower the loss early is to predict background everywhere; therefore weight
each class by the inverse of its frequency.

That argument is plausible and it is often right. **We are going to test it, and
on this dataset it is wrong**(!) for a reason that turns out to be the most useful
thing in this notebook. So we run the comparison twice: once with a model that is
structurally incapable of seeing context, and once with the full U-Net.

In [ ]:
# Per-class IoU / recall / precision, and a printer for them. Standard
# bookkeeping -- see `ms.metrics.seg_metrics??` if you want the three lines.
seg_metrics = ms.seg_metrics
show_metrics = ms.show_metrics

## 6. Experiment 1 — imbalance with a blind model

First, a model that *cannot* use context: a stack of $1\times1$ convolutions. It
has a receptive field of exactly one pixel, so it must decide "background, track,
or shower" from that pixel's charge alone. 

Noise hits carry charge in the range 1–6; the diffuse outskirts of a shower
overlap with that range. So for this model the classes genuinely are confusable.

In [ ]:
def pixel_model():
    """Receptive field = 1 pixel, by construction."""
    return nn.Sequential(nn.Conv2d(1, 32, 1), nn.ReLU(),
                         nn.Conv2d(32, 32, 1), nn.ReLU(),
                         nn.Conv2d(32, 3, 1))


def train_seg_only(model, weight=None, epochs=4, lr=3e-3, bs=64):
    torch.manual_seed(0)
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    w = None if weight is None else weight.to(DEVICE)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(TR["x"]))
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = F.cross_entropy(model(TR["x"][b].to(DEVICE)),
                                   TR["seg"][b].to(DEVICE), weight=w)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = torch.cat([model(VA["x"][i:i + 128].to(DEVICE)).argmax(1).cpu()
                          for i in range(0, len(VA["x"]), 128)])
    return pred


for tag, w in [("unweighted", None), ("inverse-frequency weighted", class_weights)]:
    pred = train_seg_only(pixel_model(), w)
    pf = np.bincount(pred.numpy().ravel(), minlength=3) / pred.numel()
    show_metrics(seg_metrics(pred, VA["seg"]), f"1x1 model, {tag}")
    print("  predicted class fractions: " +
          ", ".join(f"{ms.SEG_NAMES[c]}={pf[c]:.4f}" for c in range(3)))

**This is the textbook failure, and it is dramatic.** With unweighted
cross-entropy the blind model predicts the track class for *zero* pixels. Recall
is not low, it is exactly 0.000 — the class has been deleted from the model's
vocabulary. Tracks are the rarest class and the least separable by charge alone,
so the optimiser correctly concluded that never saying "track" is the best
available strategy under that loss.

Weighting rescues it: recall on tracks jumps to ~0.7. Precision is poor and IoU
is still bad in absolute terms, because the model genuinely does not have the
information — but it is no longer silently ignoring an entire physics class,
which is the difference between a bad model and a dangerous one.

So the standard argument is real. Now watch what happens when the model can see.

## 7. Experiment 2 — the same loss, a model with a receptive field

Now the real model: the multi-task U-Net, receptive field 76 px. Its multi-task
loss is a weighted sum:

$$\mathcal{L} = \lambda_{\text{cls}}\,\mathcal{L}_{\text{CE}}
             + \lambda_{\text{seg}}\,\mathcal{L}_{\text{CE, pixel}}
             + \lambda_{\text{E}}\,\mathcal{L}_{\text{Huber}}$$

Those $\lambda$ are real hyperparameters and getting them badly wrong will make
one task swamp the others. The practical rule: **look at the individual loss
terms during training** and make sure none of them is orders of magnitude larger
than the rest. We standardised the energy target for exactly this reason.

In [ ]:
def evaluate(model, D, bs=128):
    model.eval()
    cls_ok, seg_pred, seg_true, e_pred = 0, [], [], []
    with torch.no_grad():
        for i in range(0, len(D["x"]), bs):
            x = D["x"][i:i + bs].to(DEVICE)
            c, s, e = model(x)
            cls_ok += (c.argmax(1).cpu() == D["cls"][i:i + bs]).sum().item()
            seg_pred.append(s.argmax(1).cpu())
            seg_true.append(D["seg"][i:i + bs])
            e_pred.append(e.cpu())
    seg_pred, seg_true = torch.cat(seg_pred), torch.cat(seg_true)
    e_pred = torch.cat(e_pred)
    e_mae = (e_pred - D["energy"]).abs().mean().item() * E_STD
    return {
        "cls_acc": cls_ok / len(D["x"]),
        "seg": seg_metrics(seg_pred, seg_true),
        "energy_mae": e_mae,
        "seg_pred": seg_pred,
    }


def train_multitask(seg_weight=None, epochs=8, lr=2e-3, bs=64, tag=""):
    torch.manual_seed(0)
    model = MultiTaskNet().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    w = None if seg_weight is None else seg_weight.to(DEVICE)
    hist = []
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(TR["x"]))
        acc = np.zeros(3)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            x = TR["x"][b].to(DEVICE)
            c, s, e = model(x)
            l_cls = F.cross_entropy(c, TR["cls"][b].to(DEVICE))
            l_seg = F.cross_entropy(s, TR["seg"][b].to(DEVICE), weight=w)
            l_e = F.huber_loss(e, TR["energy"][b].to(DEVICE))
            loss = l_cls + l_seg + l_e
            opt.zero_grad(); loss.backward(); opt.step()
            acc += np.array([l_cls.item(), l_seg.item(), l_e.item()]) * len(b)
        acc /= len(perm)
        hist.append(acc)
        print(f"  {tag} epoch {ep + 1}/{epochs}  cls {acc[0]:.3f}  "
              f"seg {acc[1]:.3f}  energy {acc[2]:.4f}")
    print(f"  {tag} trained in {time.time() - t0:.0f} s")
    return model, np.array(hist)


print("RUN A — plain (unweighted) pixel cross-entropy")
model_a, hist_a = train_multitask(seg_weight=None, tag="A")

In [ ]:
res_a = evaluate(model_a, VA)
print(f"event classification accuracy : {res_a['cls_acc']:.3f}")
print(f"energy MAE                    : {res_a['energy_mae']:.3f} (target units)")
show_metrics(res_a["seg"], "segmentation, RUN A (unweighted CE)")

pred_frac = np.bincount(res_a["seg_pred"].numpy().ravel(), minlength=3) / res_a["seg_pred"].numel()
print(f"\nfraction of pixels predicted as each class: "
      f"{', '.join(f'{ms.SEG_NAMES[c]}={pred_frac[c]:.4f}' for c in range(3))}")
print(f"true fractions were:                        "
      f"{', '.join(f'{ms.SEG_NAMES[c]}={frac[c]:.4f}' for c in range(3))}")

This is not what the standard argument predicted.

The unweighted model has **not** collapsed onto the majority class. Track IoU is
~0.93; the predicted class fractions match the true ones to three decimal places.
Nothing was ignored, nothing was swamped, and no reweighting was applied.

Why? Because with a 76-pixel receptive field the classes are no longer
confusable. A noise hit is an isolated pixel; a track is a long thin connected
object with a Bragg peak; a shower is a widening cone. Those are trivially
different *given context*, and the model has context. The 98:2 ratio never
mattered, because the loss was never in a regime where predicting background was
a tempting compromise.

Now apply the standard remedy anyway, exactly as a recipe would tell you to.

In [ ]:
print("RUN B — inverse-frequency weighted pixel cross-entropy")
model_b, hist_b = train_multitask(seg_weight=class_weights, tag="B")

In [ ]:
res_b = evaluate(model_b, VA)
print(f"event classification accuracy : {res_b['cls_acc']:.3f}")
print(f"energy MAE                    : {res_b['energy_mae']:.3f} (target units)")
show_metrics(res_a["seg"], "RUN A — unweighted")
show_metrics(res_b["seg"], "RUN B — inverse-frequency weighted")

### The recipe made it worse

Weighting costs about 0.1 of IoU on both signal classes. Recall goes up — of
course it does, we told the optimiser that missing a track is ~200 times more
expensive than inventing one, and it believed us... but precision falls further
than recall rises. The model now scatters track labels over pixels that are not
tracks.

Glance at the **energy MAE** in both runs as well. The energy head was not
touched, but it shares an encoder with a segmentation loss whose scale we just
multiplied by a large factor, so it is entirely possible for it to move. When it
does, that is **multi-task interference**. 

### What to actually take away

> **Class imbalance is not, by itself, the problem. Confusability is.**
>
> Imbalance is fatal when your model cannot separate the classes on the evidence
> available to it. Then the prior dominates and the rare class disappears
> (Experiment 1). When the model *can* separate them, a 98:2 ratio costs nothing
> (Experiment 2).
>
> So when a rare class vanishes, the first question is not "which reweighting
> scheme?" It is **"can my model see the difference at all?"** Reweighting treats
> the symptom. A larger receptive field, a better representation, or a more
> informative input treats the cause. In Experiment 1 no amount of
> reweighting would have produced a good model, because the information simply
> was not there.

None of which means reweighting is useless. It means it is a **choice of
operating point**, and the right operating point comes from the physics downstream:

- reconstructing a track trajectory? Precision matters.
- searching for a rare decay you must not miss? Recall matters.

Two refinements worth knowing when you do need them:

- **Focal loss** down-weights pixels the model already classifies confidently, so
  the gradient concentrates on genuinely hard, ambiguous pixels rather than on
  every pixel of a rare class.
- **Dice / IoU loss** optimises region overlap directly instead of optimising a
  per-pixel proxy and hoping it correlates.

In [ ]:
idx = [3, 11, 19, 27]
fig, axes = plt.subplots(4, len(idx), figsize=(2.3 * len(idx), 9.2))
from matplotlib.colors import ListedColormap
cmap = ListedColormap(ms.SEG_COLORS)
for k, i in enumerate(idx):
    ms.plot_event(val, i, axes[0, k], "charge")
    axes[1, k].imshow(val["seg"][i], cmap=cmap, vmin=0, vmax=2, origin="lower")
    axes[2, k].imshow(res_a["seg_pred"][i], cmap=cmap, vmin=0, vmax=2, origin="lower")
    axes[3, k].imshow(res_b["seg_pred"][i], cmap=cmap, vmin=0, vmax=2, origin="lower")
    for r in range(1, 4):
        axes[r, k].set_xticks([]); axes[r, k].set_yticks([])
for r, name in enumerate(["input charge", "truth", "RUN A unweighted", "RUN B weighted"]):
    axes[r, 0].set_ylabel(name, fontsize=9)
fig.tight_layout(); plt.show()

## 8. The loss is part of the optimisation problem

The energy head has been trained with a **Huber** loss rather than the obvious
mean squared error. That deserves an explanation:

> A loss function is not a scoreboard. It is the surface  that gradient descent
> has to walk across. Two losses with the same minimum can  have completely
> different landscapes away from it.

Compare the gradients.

In [ ]:
r = np.linspace(-6, 6, 400)
delta = 1.0
losses = {
    "MSE  (L2)":  (r ** 2, 2 * r),
    "MAE  (L1)":  (np.abs(r), np.sign(r)),
    "Huber":      (np.where(np.abs(r) <= delta, 0.5 * r ** 2,
                            delta * (np.abs(r) - 0.5 * delta)),
                   np.clip(r, -delta, delta)),
}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for name, (l, g) in losses.items():
    axes[0].plot(r, l, label=name)
    axes[1].plot(r, g, label=name)
axes[0].set_title("loss vs residual", fontsize=10); axes[0].set_ylim(0, 12)
axes[1].set_title("GRADIENT vs residual  <- this is what training feels", fontsize=10)
for a in axes:
    a.set_xlabel("residual  (prediction - truth)"); a.grid(alpha=0.3); a.legend()
plt.show()

The left panel is the one people look at. The right panel is the one that
matters.

**The MSE gradient is unbounded and grows linearly with the residual.** Early in
training every residual is large, so the gradient is large. This is not always a
problem, and in fact it is often fine.

The problem is what happens to a *single* pathological example. An event with a
mis-simulated energy, a detector artefact, or a mislabelled truth value produces
a residual of 10 and therefore contributes 100× the gradient of a well-fit
example with residual 1. In a batch of 64, one such event can dominate the
update. Your loss curve spikes; sometimes it never recovers.

**Huber's gradient saturates.** Beyond $\delta$ it behaves like L1: a constant
pull in the right direction, no matter how absurd the residual. Outliers still
influence the fit; they can no longer hijack it.

This is exactly the reasoning behind robust estimators in a fitting problem, and
it is the same reasoning that motivates **gradient clipping**, which we meet in
Notebook 2. Clipping is what you do when you cannot change the loss.

Let us contaminate the labels and watch it happen.

In [ ]:
class EnergyOnly(nn.Module):
    """A small encoder + sum-pooled energy head, for a fast controlled test."""

    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(conv_bn_relu(1, 16), nn.MaxPool2d(2),
                                 conv_bn_relu(16, 32), nn.MaxPool2d(2),
                                 conv_bn_relu(32, 32), nn.MaxPool2d(2))
        self.head = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        return self.head(self.enc(x).sum(dim=(2, 3)) / 100.0).squeeze(1)


# 2% of training events get a corrupted energy label -- a mis-simulated sample,
# a units bug, a bad calibration constant. Entirely realistic.
y_clean = TR["energy"].clone()
y_dirty = TR["energy"].clone()
g = torch.Generator().manual_seed(0)
bad = torch.randperm(len(y_dirty), generator=g)[:int(0.02 * len(y_dirty))]
y_dirty[bad] += torch.randn(len(bad), generator=g) * 15.0
print(f"corrupted {len(bad)} of {len(y_dirty)} training labels")


def train_energy(loss_name, y, epochs=8, bs=64, lr=1e-3):
    torch.manual_seed(0)
    model = EnergyOnly().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    fn = F.mse_loss if loss_name == "mse" else F.huber_loss
    curve = []
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(TR["x"]))
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            out = model(TR["x"][b].to(DEVICE))
            loss = fn(out, y[b].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pv = torch.cat([model(VA["x"][i:i + 256].to(DEVICE)).cpu()
                            for i in range(0, len(VA["x"]), 256)])
        mae = (pv - VA["energy"]).abs().mean().item() * E_STD
        curve.append(mae)
        print(f"  {loss_name:5s} epoch {ep + 1}  clean-validation MAE = {mae:.3f}")
    return curve


print("MSE on contaminated labels")
c_mse = train_energy("mse", y_dirty)
print("\nHuber on contaminated labels")
c_hub = train_energy("huber", y_dirty)

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(range(1, len(c_mse) + 1), c_mse, "o-", label="MSE")
ax.plot(range(1, len(c_hub) + 1), c_hub, "s-", label="Huber")
ax.set_xlabel("epoch"); ax.set_ylabel("validation MAE on CLEAN labels")
ax.set_title("2% of training labels corrupted", fontsize=10)
ax.legend(); ax.grid(alpha=0.3)
plt.show()

Both models see identical data. The only difference is the shape of the loss far
from the optimum, and it is worth a substantial factor in validation error.

## 9. The encoder is the reusable part

- **Multi-task learning**, which we just did. Three supervision signals train one
  set of features, and the segmentation labels in particular are an enormously
  richer signal than a single class label per event.
  
- **Transfer learning**, which is Notebook 4. If the encoder holds general
  features, you can train it once on abundant simulation and reuse it on the
  small, precious, hand-labelled sample you actually care about.

## 10. Takeaways

1. **Compute your receptive field before you train.** A network that cannot see
   the evidence will not find it, however long you train it.
2. **Downsampling is how a CNN acquires context**, not merely how it saves
   compute — and it costs resolution.
3. **U-Net skips restore resolution; residual skips fix gradients.** Different
   mechanisms, same nickname.
4. **One encoder, many heads.** Match each head's pooling to the physics of its
   target: max for existence, sum for extensive quantities, none for per-pixel.
   (With the caveat from §3: on a *fixed-size* input, sum and mean pooling differ
   only by a constant the next layer absorbs. The choice is a real prior only when
   the number of pooled elements varies -- see Notebook 3.)
5. **Imbalance is not the problem; confusability is.** A rare class disappears
   when your model cannot separate it on the evidence available (the $1\times1$
   model: track recall exactly 0.000). Give the model context and a 98:2 ratio
   costs nothing. Ask "can it see the difference?" before you ask "which
   reweighting scheme?"
6. **Class weighting is a choice of operating point, not a bug fix**, and applied
   reflexively here it cost ~0.1 of IoU on both signal classes.
7. **Report IoU / recall / precision, never bare pixel accuracy**, when one class
   dominates.
8. **The gradient of your loss is what training actually feels.** Unbounded
   gradients hand control of your batch to your worst-simulated event.
9. **Textbook advice is a hypothesis about your data.** Two short runs settled
   it. Run the ablation.

## 11. Exercises

1. **Break the receptive field.** Retrain the multi-task model with only two
   encoder blocks. Which of the three tasks degrades most, and does the receptive
   field table explain why?
2. **Wrong pooling on purpose.** Change the energy head from `sum` to `mean`
   pooling and retrain. Predict what happens to events containing two tracks
   before you run it.
3. **Remove the U-Net skips** (feed only the upsampled features to each decoder
   block). Classification should barely move; segmentation should get visibly
   blurry. Why?
4. **Try focal loss** for segmentation
   ($\mathcal{L} = -(1-p_t)^\gamma \log p_t$, $\gamma \approx 2$) and compare its
   precision/recall trade-off against inverse-frequency weighting.
5. **Single-task baselines.** Train classification alone, with no segmentation or
   energy loss. Is the multi-task model better? By how much? Multi-task learning
   is usually helpful, but it is not free — the tasks compete for capacity.